<a href="https://colab.research.google.com/github/feliciadominic14-create/Homework/blob/main/Felicia_Dominic_QuickBite_Student_ICTAK_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study: QuickBite Food Delivery Analysis
**ICT Academy of Kerala · Python Programming · pandas · 1-hour case study · Beginner to intermediate**

**Student version.** Write your code in the empty cells. The expected output is given under each task.

## The scenario
**QuickBite** is a food delivery start-up operating in four Kerala cities. The operations manager has exported the orders of the last **3 months** (August to October 2026) and the list of partner restaurants as two CSV files. The orders file was typed from several systems, so it contains **duplicate rows, inconsistent text, missing values and impossible delivery times**.

Your job is to **clean the data, answer the manager's questions and produce a short report** using pandas.

## Files
Keep these two files **in the same folder as this notebook** (in Google Colab, upload them with the Files panel on the left).

| File | Contents |
|---|---|
| `quickbite_orders.csv` | order_id, customer_id, restaurant_id, city, order_time, items, amount (₹), payment_mode, delivery_minutes, rating, status |
| `quickbite_restaurants.csv` | restaurant_id, restaurant_name, cuisine |

## How this case study works
| Stage | Level | Tasks | Time |
|---|---|---|---|
| A · Load and inspect | Beginner | 1 – 2 | 6 min |
| B · Clean the data | Beginner | 3 – 6 | 14 min |
| C · Analyse | Intermediate | 7 – 12 | 18 min |
| D · Insights | Intermediate | 13 – 15 | 10 min |
| E · Report | Intermediate | 16 – 17 | 8 min |
| Challenge | Optional | 18 | 4 min |

**Rules**
- Use **pandas** (and NumPy where needed). Do not edit the CSV files by hand.
- **Tasks share variables**, so run the cells in order from the top.
- Your output should **match the expected output** (it may look like a formatted table; the values must match).
- Finish with **Kernel → Restart and Run All** and make sure there are no errors.
- Text columns may show type `str` or `object` depending on your pandas version. Both are fine.

Run this cell first.

In [19]:
import pandas as pd
import numpy as np
from google.colab import files
import io

uploaded = files.upload()

restaurant_file = [f for f in uploaded if f.startswith("quickbite_restaurants")][0]
orders_file = [f for f in uploaded if f.startswith("quickbite_orders")][0]

df1 = pd.read_csv(io.BytesIO(uploaded[restaurant_file]))
df2 = pd.read_csv(io.BytesIO(uploaded[orders_file]))

print("Restaurants dataset:", df1.shape)
print("Orders dataset:", df2.shape)

print("\nRestaurants:")
display(df1.head())

print("\nOrders:")
display(df2.head())


Saving quickbite_restaurants.csv to quickbite_restaurants (8).csv
Saving quickbite_orders.csv to quickbite_orders (8).csv
Restaurants dataset: (12, 3)
Orders dataset: (405, 11)

Restaurants:


,restaurant_id,restaurant_name,cuisine
0,R01,Spice Garden,North Indian
1,R02,Dosa Corner,South Indian
2,R03,Wok Express,Chinese
3,R04,Pizza Point,Italian
4,R05,Biryani House,Biryani



Orders:


,order_id,customer_id,restaurant_id,city,order_time,items,amount,payment_mode,delivery_minutes,rating,status
0,QB1349,C020,R02,Kochi,2026-10-22 22:00,3,516.0,UPI,32.0,4.0,Delivered
1,QB1002,C046,R02,Kozhikode,2026-10-05 17:00,3,457.0,Wallet,40.0,4.0,Delivered
2,QB1393,C058,R01,TRIVANDRUM,2026-08-24 18:00,1,215.0,Wallet,31.0,5.0,Delivered
3,QB1120,C018,R06,Kochi,2026-10-29 17:00,3,563.0,UPI,39.0,4.0,Delivered
4,QB1155,C021,R05,Kozhikode,2026-09-19 18:00,5,1075.0,UPI,NaN,NaN,Cancelled


---
## Stage A · Load and inspect

### Task 1 · Load the data
Load `quickbite_orders.csv` into `orders` and `quickbite_restaurants.csv` into `restaurants`. Print the shape of each (orders first).

**Expected output**
```
(405, 11)
(12, 3)
```

In [20]:

orders = pd.read_csv("quickbite_orders.csv")
restaurants = pd.read_csv("quickbite_restaurants.csv")

print(orders.shape)
print(restaurants.shape)


(405, 11)
(12, 3)


### Task 2 · Find the problems
Print the columns of `orders` that have missing values together with the number missing (only columns with at least one missing value), and then the number of fully duplicated rows.

**Expected output**
```
delivery_minutes    60
rating              67
dtype: int64
5
```

In [21]:
print(orders.isna().sum()[orders.isna().sum() > 0])
print(orders.duplicated().sum())


delivery_minutes    60
rating              67
dtype: int64
5


---
## Stage B · Clean the data

### Task 3 · Remove duplicates
Remove the duplicate rows from `orders` (keep the first of each) and reset the row numbers. Print the new shape.

**Expected output**
```
(400, 11)
```

In [22]:
orders = orders.drop_duplicates(keep="first").reset_index(drop=True)
print(orders.shape)

(400, 11)


### Task 4 · Make the text consistent
The `payment_mode` and `city` columns have spaces and mixed upper/lower case. Strip the spaces and convert both to title case. Print the number of orders for each payment mode (sorted by the payment mode name) and then for each city (sorted by the city name).

**Expected output**
```
payment_mode
Card       98
Cash       75
Upi       184
Wallet     43
Name: count, dtype: int64
city
Kochi         174
Kozhikode      73
Thrissur       40
Trivandrum    113
Name: count, dtype: int64
```

In [23]:
orders["payment_mode"] = orders["payment_mode"].str.strip().str.title()
orders["city"] = orders["city"].str.strip().str.title()

print(orders["payment_mode"].value_counts().sort_index())
print(orders["city"].value_counts().sort_index())

payment_mode
Card       98
Cash       75
Upi       184
Wallet     43
Name: count, dtype: int64
city
Kochi         174
Kozhikode      73
Thrissur       40
Trivandrum    113
Name: count, dtype: int64


### Task 5 · Fix delivery times
A delivery can never take more than 180 minutes, so larger values are typing errors. Replace them with missing values and print how many you replaced. Then fill the missing `delivery_minutes` of **Delivered** orders with the median delivery time of delivered orders (cancelled orders stay missing). Print the median and the number of missing `delivery_minutes` left.

**Expected output**
```
4
34.5
49
```

In [24]:
invalid_times = orders["delivery_minutes"] > 180
print(invalid_times.sum())
orders.loc[invalid_times, "delivery_minutes"] = np.nan
delivered_median = orders.loc[orders["status"] == "Delivered", "delivery_minutes"].median()
print(delivered_median)
mask = (orders["status"] == "Delivered") & (orders["delivery_minutes"].isna())
orders.loc[mask, "delivery_minutes"] = delivered_median
print(orders["delivery_minutes"].isna().sum())

4
34.5
49


### Task 6 · Work with dates
Convert `order_time` to a real date-time. Add three columns: `month_no` (month number), `weekday` (day name such as Monday) and `hour`. Print the `order_time`, `month_no`, `weekday` and `hour` of the first 3 rows.

**Expected output**
```
           order_time  month_no   weekday  hour
0 2026-10-22 22:00:00        10  Thursday    22
1 2026-10-05 17:00:00        10    Monday    17
2 2026-08-24 18:00:00         8    Monday    18
```

In [25]:
orders["order_time"] = pd.to_datetime(orders["order_time"])
orders["month_no"] = orders["order_time"].dt.month
orders["weekday"] = orders["order_time"].dt.day_name()
orders["hour"] = orders["order_time"].dt.hour

print(orders[["order_time", "month_no", "weekday", "hour"]].head(3))

           order_time  month_no   weekday  hour
0 2026-10-22 22:00:00        10  Thursday    22
1 2026-10-05 17:00:00        10    Monday    17
2 2026-08-24 18:00:00         8    Monday    18


---
## Stage C · Analyse

### Task 7 · Add restaurant details
Create `df` by merging `orders` with `restaurants` on `restaurant_id`, keeping every order. Print the shape of `df` and the number of orders with no `restaurant_name`.

**Expected output**
```
(400, 16)
0
```

In [26]:
df = orders.merge(restaurants, on="restaurant_id", how="left")
print(df.shape)
print(df["restaurant_name"].isna().sum())

(400, 16)
0


### Task 8 · Cancellations
Print the overall cancellation percentage (1 decimal). Then print the cancellation percentage of each city (1 decimal), from highest to lowest.

**Expected output**
```
12.2
city
Trivandrum    13.3
Kochi         12.6
Kozhikode     11.0
Thrissur      10.0
Name: status, dtype: float64
```

In [27]:
overall_cancel_pct = df["status"].eq("Cancelled").mean() * 100
print(round(overall_cancel_pct, 1))

city_cancel_pct = (
    df.groupby("city")["status"]
      .apply(lambda s: s.eq("Cancelled").mean() * 100)
      .sort_values(ascending=False)
)
print(city_cancel_pct.round(1))

12.2
city
Trivandrum    13.3
Kochi         12.6
Kozhikode     11.0
Thrissur      10.0
Name: status, dtype: float64


### Task 9 · Revenue
From now on use only **Delivered** orders: create `delivered` for them. Print the total revenue (sum of `amount`) and the average order value (1 decimal).

**Expected output**
```
205798.0
586.3
```

In [28]:
delivered = df[df["status"] == "Delivered"].copy()
print(delivered["amount"].sum())
print(round(delivered["amount"].mean(), 1))

205798.0
586.3


### Task 10 · Cuisine revenue
Print the total revenue of each cuisine for the 3 best cuisines, from highest to lowest.

**Expected output**
```
cuisine
South Indian    51789.0
Chinese         30275.0
Biryani         29365.0
Name: amount, dtype: float64
```

In [29]:
cuisine_revenue = delivered.groupby("cuisine")["amount"].sum()
print(cuisine_revenue.nlargest(3))

cuisine
South Indian    51789.0
Chinese         30275.0
Biryani         29365.0
Name: amount, dtype: float64


### Task 11 · Best restaurants
Create a table with one row per restaurant and the columns `orders` (number of orders) and `revenue` (sum of amount). Print the top 5 restaurants by revenue.

**Expected output**
```
                 orders  revenue
restaurant_name                 
Biryani House        48  29365.0
Dosa Corner          50  26812.0
Malabar Kitchen      41  24977.0
Pizza Point          32  20038.0
Wok Express          28  18618.0
```

In [30]:
restaurant_summary = delivered.groupby("restaurant_name").agg(
    orders=("order_id", "count"),
    revenue=("amount", "sum")
)
print(restaurant_summary.sort_values("revenue", ascending=False).head(5))

                 orders  revenue
restaurant_name                 
Biryani House        48  29365.0
Dosa Corner          50  26812.0
Malabar Kitchen      41  24977.0
Pizza Point          32  20038.0
Wok Express          28  18618.0


### Task 12 · City summary
For delivered orders print one table with a row per city and the columns `orders`, `avg_delivery` and `avg_rating`, rounded to 1 decimal.

**Expected output**
```
            orders  avg_delivery  avg_rating
city                                        
Kochi          152          34.6         3.8
Kozhikode       65          33.9         3.9
Thrissur        36          38.2         4.2
Trivandrum      98          32.9         4.0
```

In [31]:
city_summary = delivered.groupby("city").agg(
    orders=("order_id", "count"),
    avg_delivery=("delivery_minutes", "mean"),
    avg_rating=("rating", "mean")
)
print(city_summary.round(1))

            orders  avg_delivery  avg_rating
city                                        
Kochi          152          34.6         3.8
Kozhikode       65          33.9         3.9
Thrissur        36          38.2         4.2
Trivandrum      98          32.9         4.0


---
## Stage D · Insights

### Task 13 · Busiest day
Using **all** orders in `df`, print the number of orders for each weekday, from the busiest to the quietest.

**Expected output**
```
weekday
Monday       70
Thursday     57
Sunday       57
Wednesday    57
Saturday     56
Tuesday      53
Friday       50
Name: count, dtype: int64
```

In [32]:
print(df["weekday"].value_counts())

weekday
Monday       70
Thursday     57
Sunday       57
Wednesday    57
Saturday     56
Tuesday      53
Friday       50
Name: count, dtype: int64


### Task 14 · Pivot table
Print a pivot table of total revenue (delivered orders) with `city` as rows and `payment_mode` as columns, rounded to whole numbers.

**Expected output**
```
payment_mode     Card     Cash      Upi   Wallet
city                                            
Kochi         20029.0  17044.0  40501.0  11017.0
Kozhikode     10023.0   5322.0  21722.0   1760.0
Thrissur       6248.0   2954.0  10907.0   1051.0
Trivandrum    11522.0   8606.0  27555.0   9537.0
```

In [33]:
revenue_pivot = pd.pivot_table(
    delivered,
    values="amount",
    index="city",
    columns="payment_mode",
    aggfunc="sum",
    fill_value=0
)
print(revenue_pivot.round(0))

payment_mode     Card     Cash      Upi   Wallet
city                                            
Kochi         20029.0  17044.0  40501.0  11017.0
Kozhikode     10023.0   5322.0  21722.0   1760.0
Thrissur       6248.0   2954.0  10907.0   1051.0
Trivandrum    11522.0   8606.0  27555.0   9537.0


### Task 15 · Monthly trend and order size
Print the total revenue of each month number (delivered orders). Then create `size` with `pd.cut` on `items` using bins `[0, 2, 4, 6]` and labels `Small`, `Medium`, `Large`, and print the average amount of each size (1 decimal).

**Expected output**
```
month_no
8     76536.0
9     60517.0
10    68745.0
Name: amount, dtype: float64
size
Small     279.5
Medium    572.2
Large     919.5
Name: amount, dtype: float64
```

In [34]:
monthly_revenue = delivered.groupby("month_no")["amount"].sum()
print(monthly_revenue)

delivered["size"] = pd.cut(
    delivered["items"],
    bins=[0, 2, 4, 6],
    labels=["Small", "Medium", "Large"]
)
print(delivered.groupby("size", observed=False)["amount"].mean().round(1))

month_no
8     76536.0
9     60517.0
10    68745.0
Name: amount, dtype: float64
size
Small     279.5
Medium    572.2
Large     919.5
Name: amount, dtype: float64


---
## Stage E · Report

### Task 16 · Print the report
Print a table of the top 5 restaurants by revenue: headings `Rank` (left, 6), `Restaurant` (left, 18), `Orders` (right, 8) and `Revenue` (right, 12), a line of 44 dashes and one row per restaurant. Show revenue with a comma as thousands separator and no decimals.

**Expected output**
```
Rank  Restaurant          Orders     Revenue
--------------------------------------------
1     Biryani House           48      29,365
2     Dosa Corner             50      26,812
3     Malabar Kitchen         41      24,977
4     Pizza Point             32      20,038
5     Wok Express             28      18,618
```

In [35]:
top5 = restaurant_summary.sort_values("revenue", ascending=False).head(5)
print(f"{'Rank':<6}{'Restaurant':<18}{'Orders':>8}{'Revenue':>12}")
print("-" * 44)

for rank, (restaurant, row) in enumerate(top5.iterrows(), start=1):
    print(f"{rank:<6}{restaurant:<18}{int(row['orders']):>8}{row['revenue']:>12,.0f}")

Rank  Restaurant          Orders     Revenue
--------------------------------------------
1     Biryani House           48      29,365
2     Dosa Corner             50      26,812
3     Malabar Kitchen         41      24,977
4     Pizza Point             32      20,038
5     Wok Express             28      18,618


### Task 17 · Save the clean data
Save `df` as `quickbite_clean.csv` without the index column. Load the saved file again and print its shape and the first 3 `order_id` values.

**Expected output**
```
(400, 16)
['QB1349', 'QB1002', 'QB1393']
```

In [36]:
df.to_csv("quickbite_clean.csv", index=False)
clean_check = pd.read_csv("quickbite_clean.csv")
print(clean_check.shape)
print(clean_check["order_id"].head(3).tolist())

(400, 16)
['QB1349', 'QB1002', 'QB1393']


---
## Stage Challenge (optional)

### Task 18 · Repeat customers
A **repeat customer** has placed **5 or more** orders (all orders in `df`). Print the number of repeat customers and what percentage of all customers they are (1 decimal).

**Expected output**
```
47
60.3
```

In [ ]:
# Write your code here


---
## Submission
Save the notebook as **`YourName_QuickBite_CaseStudy.ipynb`**, run **Restart and Run All**, and submit as instructed by your trainer.

**Reflect:** which task took you the longest? Which pandas functions did you have to look up? Practise those next.